# Capstone Project: End-to-End MLOps Pipeline
### Phase 1: Data Ingestion and Exploration

This notebook loads a binary classification dataset into Databricks, stores the dataset in Unity Catalog Volume, performs exploratory data analysis using PySpark DataFrames and the Pandas API on Spark, and documents the schema, feature distributions, and preprocessing plan.

Dataset: Breast Cancer Wisconsin classification dataset  
Target column: `target`  
Problem type: Binary classification 


In [0]:
# Upload Dataset to Unity Catalog Volume

from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load the data
data = load_breast_cancer(as_frame=True)
df = data.frame

# Replace spaces with underscores in column names
df.columns = df.columns.str.replace(' ', '_')

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Use current catalog and default schema
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
table_name = f"{current_catalog}.default.breast_cancer"

# Create schema if needed
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {current_catalog}.default")

# Drop table if exists to start fresh
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Write to table
spark_df.write.saveAsTable(table_name)

print(f"Data saved to Unity Catalog table: {table_name}")
display(spark_df.head())

In [0]:
# Read from Unity Catalog table (created in cell 2)

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
spark_df = spark.table(f"{current_catalog}.default.breast_cancer")

display(spark_df)
spark_df.printSchema()

In [0]:
# EDA with pyspark

from pyspark.sql.functions import col, count, isnan, when

# Summary statistics
display(spark_df.describe())

# Missing values by column
missing_df = spark_df.select([
    count(
        when(
            col(c).isNull() | isnan(col(c)),
            c
        )
    ).alias(c)
    for c in spark_df.columns
])

display(missing_df)

# Target distribution
display(spark_df.groupBy("target").count())

In [0]:
# EDA with Pandas API on Spark

import pyspark.pandas as ps

psdf = spark_df.pandas_api()

display(psdf.head())
display(psdf.describe())

In [0]:
# Visual Distribution Plots

import matplotlib.pyplot as plt

sample_pdf = spark_df.toPandas()

selected_columns = [
    "mean_radius",
    "mean_texture", 
    "mean_perimeter",
    "mean_area",
    "target"
]

sample_pdf[selected_columns].hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

## Data Schema

The dataset contains numeric tumor measurements and a binary target column.

- Target: `target`
- Type: Binary classification
- Classes:
  - 0 = malignant
  - 1 = benign

## Preprocessing

- Checked missing values.
- Used numeric features directly.
- Split into train/test sets.
- Standardized features using `StandardScaler`.